# Lab 06 challenge: worked solution

For the instructor. It repeats the pieces of the lab it needs (the label, the steps, the workflow, the agent, the
meter and the checks) so it runs on its own, then solves the two tasks. Values in the text come from the run saved
with this notebook (`gpt-4o-mini`, 13 Oct 2026).

In [1]:
# Setup: on Colab, copy the repository and install the packages. Locally (after uv sync) nothing is installed.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/agentic_ai_conversational_agents").exists():
        !git clone -q --depth 1 https://github.com/tatwan/agentic_ai_conversational_agents.git /content/agentic_ai_conversational_agents
    %cd /content/agentic_ai_conversational_agents
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: agentic_ai_conversational_agents


In [2]:
# Config: model names come from models.env; the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name in ("CHAT_MODEL", "REASONING_EFFORT", "JUDGE_MODEL", "EMBED_MODEL"):
    print(f"{name:18}{settings[name] or '(not set)'}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

CHAT_MODEL        gpt-4o-mini
REASONING_EFFORT  (not set)
JUDGE_MODEL       gpt-4o-mini
EMBED_MODEL       text-embedding-3-small
OPENAI_API_KEY    found in .env


## Setup: the pieces from the lab

Run this cell and move on; the answers start at Task 1.

In [3]:
import json
import re
from typing import Literal
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_core.callbacks import UsageMetadataCallbackHandler
from shared.settings import chat_model
from shared.ops_api import start_ops_api
from shared.ops_tools import find_customer, get_customer_invoices, get_invoice

server = start_ops_api()
model = chat_model(settings)

class Request(BaseModel):
    label: Literal["policy_question", "account_lookup", "refund_request", "out_of_scope"]
    customer_name: str | None = Field(description="The customer's name, if the message names one.")
    invoice_ids: list[int] = Field(description="Invoice numbers in the message, digits only.")

labeller = model.with_structured_output(Request)

DRAFT = ("Answer the support rep's message in two or three sentences, using only these facts from the Ops API.\n"
         "Message: {text}\nFacts: {facts}")
POLICY_REPLY = "I can't read the handbook yet. Please check the policy in the support handbook."
DECLINE = "I can help with Chinook customers and invoices. That is outside what I do."

def draft(text, facts, config=None):
    return model.invoke(DRAFT.format(text=text, facts=facts), config).text

def account_facts(request):
    found = find_customer.invoke({"name": request.customer_name})
    if not found.startswith("["):
        return found
    customers = json.loads(found)
    if not customers:
        return f"No customer named {request.customer_name}."
    invoices = get_customer_invoices.invoke({"customer_id": customers[0]["customer_id"]})
    return json.dumps(customers[0]) + "\n" + invoices

def refund_facts(request):
    return "\n".join(get_invoice.invoke({"invoice_id": number}) for number in request.invoice_ids)

PROMPT = ("You are Chinook's support copilot. You help support reps with questions about customers and invoices. "
          "Use the tools to look up facts, and quote numbers exactly as the tools return them.")
agent = create_agent(model, tools=[find_customer, get_customer_invoices, get_invoice], system_prompt=PROMPT)

def ask_agent(text, config=None):
    result = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
    return result["messages"][-1].text

class Meter(UsageMetadataCallbackHandler):
    # LangChain's token counter, plus a count of model calls.
    def __init__(self):
        super().__init__()
        self.calls = 0

    def on_llm_end(self, response, **kwargs):
        super().on_llm_end(response, **kwargs)
        self.calls += 1

    def tokens(self):
        return sum(usage["total_tokens"] for usage in self.usage_metadata.values())

REQUESTS = [
    "How many invoices does Puja Srivastava have, and what is her total?",
    "Ticket 14: the customer says she was charged twice, invoices 9001 and 9002. Check them before we refund.",
    "Write me a poem about jazz.",
    "How many invoices does Helena Holý have, and which albums were on her first one?",
]

def albums(invoice_id):
    return sorted({line["album"] for line in json.loads(get_invoice.invoke({"invoice_id": invoice_id}))["lines"]})

ALBUMS_46 = albums(46)

def check(number, reply):
    text = reply.lower()
    if number == 1:
        return re.search(r"\b(6|six)\b", text) is not None and "36.64" in text
    if number == 2:
        return "9001" in text and "9002" in text and "9.99" in text
    if number == 3:
        return reply.count("\n") < 3
    return re.search(r"\b(7|seven)\b", text) is not None and all(album.lower() in text for album in ALBUMS_46)

Ops API started at http://127.0.0.1:8765


## Task 1: Add a step for the albums

One more field in the schema, and one more API call in the account branch when the field is true. The other branches
are unchanged.

In [4]:
class RequestWithInvoice(Request):
    wants_first_invoice: bool = Field(description="True when the rep asks about the customer's first invoice.")

labeller_2 = model.with_structured_output(RequestWithInvoice)

fetched = []

def account_facts_2(request):
    if not request.customer_name:         # the label found no name: tell the draft instead of calling the API
        return "The message names no customer."
    facts = account_facts(request)        # the customer, then her invoice list as JSON on the second line
    if request.wants_first_invoice and "\n" in facts:
        first = json.loads(facts.split("\n", 1)[1])["invoices"][0]["invoice_id"]
        fetched.append(first)
        facts += "\n" + get_invoice.invoke({"invoice_id": first})
    return facts

def workflow_2(text, config=None):
    request = labeller_2.invoke(text, config)
    print("   wants_first_invoice:", request.wants_first_invoice)
    if request.label == "account_lookup":
        return draft(text, account_facts_2(request), config)
    elif request.label == "refund_request":
        return draft(text, refund_facts(request), config)
    elif request.label == "policy_question":
        return POLICY_REPLY
    else:
        return DECLINE

Request 4 three times, checked as in the lab.

In [5]:
for run in range(3):
    fetched.clear()
    meter = Meter()
    reply = workflow_2(REQUESTS[3], {"callbacks": [meter]})
    named = [album for album in ALBUMS_46 if album.lower() in reply.lower()]
    print(f"run {run + 1}: fetched {fetched} | model calls {meter.calls} | albums named {len(named)} of {len(ALBUMS_46)}"
          f" | check 4 {check(4, reply)} | tokens {meter.tokens()}")
    print("  ", reply[:200].replace("\n", " "))

   wants_first_invoice: True


run 1: fetched [46] | model calls 2 | albums named 2 of 5 | check 4 False | tokens 887
   Helena Holý has a total of 7 invoices. Her first invoice, dated July 11, 2009, includes albums such as "Emergency On Planet Earth" and "Synkronized."


   wants_first_invoice: True


run 2: fetched [46] | model calls 2 | albums named 4 of 5 | check 4 False | tokens 904
   Helena Holý has a total of 7 invoices. Her first invoice, dated July 11, 2009, includes tracks from the albums "Emergency On Planet Earth" and "Synkronized," as well as "The Return Of The Space Cowboy


   wants_first_invoice: True


run 3: fetched [46] | model calls 2 | albums named 2 of 5 | check 4 False | tokens 888
   Helena Holý has a total of 7 invoices. Her first invoice, dated July 11, 2009, includes tracks from the albums "Emergency On Planet Earth" and "Synkronized."


Three checks, read apart. The label set `wants_first_invoice` to `True` and the new step fetched invoice 46 in 3 of 3
runs, at 2 model calls: the lookup worked. The replies named 2, 4 and 2 of the five albums, so check 4 failed every
run. The draft's instruction asks for two or three sentences, and the model shortened the list. One more line in the
instruction:

In [6]:
DRAFT += "\nWhen the facts include an invoice's lines, name every album on it."    # draft() reads DRAFT each call

for run in range(3):
    fetched.clear()
    meter = Meter()
    reply = workflow_2(REQUESTS[3], {"callbacks": [meter]})
    named = [album for album in ALBUMS_46 if album.lower() in reply.lower()]
    print(f"run {run + 1}: fetched {fetched} | model calls {meter.calls} | albums named {len(named)} of {len(ALBUMS_46)}"
          f" | check 4 {check(4, reply)} | tokens {meter.tokens()}")
    print("  ", reply[:220].replace("\n", " "))

   wants_first_invoice: True


run 1: fetched [46] | model calls 2 | albums named 5 of 5 | check 4 True | tokens 930
   Helena Holý has a total of 7 invoices. Her first invoice, dated July 11, 2009, included tracks from the albums *Emergency On Planet Earth* and *Synkronized*, as well as *The Return Of The Space Cowboy* and *Get Born*, al


   wants_first_invoice: True


run 2: fetched [46] | model calls 2 | albums named 5 of 5 | check 4 True | tokens 920
   Helena Holý has a total of 7 invoices. Her first invoice, dated July 11, 2009, included tracks from the albums "Emergency On Planet Earth", "Synkronized", "The Return Of The Space Cowboy", "Get Born", and "Are You Experi


   wants_first_invoice: True


run 3: fetched [46] | model calls 2 | albums named 5 of 5 | check 4 True | tokens 920
   Helena Holý has a total of 7 invoices. Her first invoice, dated July 11, 2009, includes tracks from the albums "Emergency On Planet Earth," "Synkronized," "The Return Of The Space Cowboy," "Get Born," and "Are You Experi


With that line the replies named all five albums in 3 of 3 runs, still at 2 model calls (about 920 tokens).

Now her last invoice, three times. Its albums come from the API, so the check is the same kind as request 4's.

In [7]:
LAST = "Which albums were on Helena Holý's last invoice?"
last_id = json.loads(get_customer_invoices.invoke({"customer_id": 6}))["invoices"][-1]["invoice_id"]
ALBUMS_LAST = albums(last_id)
print("last invoice:", last_id, ALBUMS_LAST)

for run in range(3):
    fetched.clear()
    meter = Meter()
    reply = workflow_2(LAST, {"callbacks": [meter]})
    named = [album for album in ALBUMS_LAST if album.lower() in reply.lower()]
    print(f"fetched {fetched} | albums of the last invoice named: {len(named)} of {len(ALBUMS_LAST)} | calls={meter.calls}")
    print("  ", reply[:200].replace("\n", " "))

last invoice: 404 ['Achtung Baby', 'Battlestar Galactica, Season 3', 'Heroes, Season 1', 'Lost, Season 1', 'Lost, Season 2', 'Lost, Season 3', 'Volume Dois']


   wants_first_invoice: False


fetched [] | albums of the last invoice named: 0 of 7 | calls=2
   Helena Holý's last invoice was dated November 13, 2013, with a total of 25.86. Unfortunately, I don't have the details of the albums included in that invoice.


   wants_first_invoice: False


fetched [] | albums of the last invoice named: 0 of 7 | calls=2
   Helena Holý's last invoice was dated November 13, 2013, with a total of 25.86. Unfortunately, the details of the albums included in that invoice are not available. Let me know if there's anything else


   wants_first_invoice: False


fetched [] | albums of the last invoice named: 0 of 7 | calls=2
   I'm sorry, but your message doesn't specify a customer. If you need details about Helena Holý's last invoice, please provide her name, and I'll be happy to assist you!


The last-invoice question failed 3 of 3. The field was `False` in every run, so the branch never fetched invoice 404;
two replies said the album details were not available, and in the third the label found no customer name, so the
guard's sentence went to the draft and the reply asked for one. What this workflow lacks is a way to select the
invoice: its only invoice step fetches the first one. That can be added in code, a field that names which invoice and
a step that fetches it, and it stays a workflow; the next kind of question would need another change. The agent
needs none: the model reads the question and chooses which invoice to look up.

## Task 2: A label in front of the agent

The label runs first; only `out_of_scope` is answered in code, and everything else goes to the agent.

In [8]:
def gated(text, config=None):
    request = labeller.invoke(text, config)
    if request.label == "out_of_scope":
        return DECLINE
    return ask_agent(text, config)

for number, text in enumerate(REQUESTS, 1):
    for shape, run in (("agent", ask_agent), ("gated", gated)):
        meter = Meter()
        reply = run(text, {"callbacks": [meter]})
        print(f"{number}. {shape:6} passed={check(number, reply)!s:5} calls={meter.calls} tokens={meter.tokens()}")

1. agent  passed=True  calls=3 tokens=984


1. gated  passed=True  calls=4 tokens=1142


2. agent  passed=True  calls=2 tokens=817


2. gated  passed=True  calls=3 tokens=994


3. agent  passed=False calls=1 tokens=347


3. gated  passed=True  calls=1 tokens=143


4. agent  passed=True  calls=4 tokens=2038


4. gated  passed=True  calls=5 tokens=2197


The poem got the fixed reply at 1 call and 143 tokens (the agent alone wrote a poem, 347 tokens). Requests 1, 2 and 4 passed as before and cost one call more each (4, 3 and 5 instead of 3, 2 and 4) and 158 to 177 more tokens. A label in front pays off only when enough requests stop at it.

In [9]:
if server is not None:
    server.should_exit = True